# Sprint 8 — Búsqueda de Hiperparámetros (HPO)
**Random Search vs Bayesian Optimization (Optuna)**

## Entregables
1. Experimentos comparables Random/Bayes con pruning/early stopping
2. Logs + artefactos guardados
3. Tabla top-k y gráfico de evolución
4. Resumen del espacio, presupuesto y decisión (config ganadora)

In [1]:
import subprocess, sys
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
script = ROOT / 'scripts' / 'semana8_hpo.py'

result = subprocess.run(
    [sys.executable, str(script)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

[2026-06-03T19:27:39] Cargando dataset PKL...
  Muestras: 3684 | Clases: 1086 | Features: 108
  Dataset MD5: 473828a489f4797b839218c8169a05e1

RANDOM SEARCH  (n_trials=10)
  Trial 01/10 | lr  | F1=0.0053 | best_so_far=0.0053
  Trial 02/10 | rf  | F1=0.0032 | best_so_far=0.0053
  Trial 03/10 | lr  | F1=0.0065 | best_so_far=0.0065
  Trial 04/10 | hgb | F1=0.0000 | best_so_far=0.0065
  Trial 05/10 | lr  | F1=0.0067 | best_so_far=0.0067
  Trial 06/10 | lr  | F1=0.0032 | best_so_far=0.0067
  Trial 07/10 | rf  | F1=0.0030 | best_so_far=0.0067
  Trial 08/10 | lr  | F1=0.0067 | best_so_far=0.0067
  Trial 09/10 | rf  | F1=0.0037 | best_so_far=0.0067
  Trial 10/10 | lr  | F1=0.0040 | best_so_far=0.0067

BAYESIAN SEARCH  (n_trials=10, pruner=MedianPruner)

STDERR: [W 2026-06-03 21:13:10,691] Trial 0 failed with parameters: {'model': 'hgb', 'lr__C': 0.6251373574521749, 'lr__max_iter': 131, 'hgb__learning_rate': 0.01699897838270077, 'hgb__max_iter': 34, 'hgb__max_leaf_nodes': 29, 'rf__n_estimators'

## Espacio de búsqueda y presupuesto

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path('..') if Path('../data').exists() else Path('.')

resumen_path = ROOT / 'data' / 'semana8_resumen.txt'
print(resumen_path.read_text())

FileNotFoundError: [Errno 2] No such file or directory: '../data/semana8_resumen.txt'

## Tabla top-k (combinado Random + Bayes)

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')

df_rand  = pd.read_csv(ROOT / 'data' / 'semana8_random_trials.csv')
df_bayes = pd.read_csv(ROOT / 'data' / 'semana8_bayes_trials.csv')

df_all = pd.concat([df_rand, df_bayes], ignore_index=True)
df_ok  = df_all[df_all['status'] == 'ok'].copy()

top5 = (
    df_ok
    .sort_values('f1_macro_mean', ascending=False)
    .head(5)
    [['method', 'model', 'f1_macro_mean', 'f1_macro_std',
      'lr__C', 'lr__max_iter',
      'hgb__learning_rate', 'hgb__max_iter', 'hgb__max_leaf_nodes',
      'rf__n_estimators', 'rf__max_depth', 'rf__min_samples_leaf']]
    .reset_index(drop=True)
)
top5.index += 1
top5.index.name = 'rank'

print('=== Top-5 configuraciones ===')
display(top5.style.format({
    'f1_macro_mean': '{:.4f}',
    'f1_macro_std':  '{:.4f}',
    'lr__C':         '{:.4f}',
    'hgb__learning_rate': '{:.4f}',
}).highlight_max(subset=['f1_macro_mean'], color='#c3efb0'))

## Gráfico de evolución y comparación

In [ ]:
from IPython.display import Image
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')
Image(str(ROOT / 'data' / 'semana8_graficos.png'), width=1000)

## Análisis: Random Search vs Bayesian

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')

df_rand  = pd.read_csv(ROOT / 'data' / 'semana8_random_trials.csv')
df_bayes = pd.read_csv(ROOT / 'data' / 'semana8_bayes_trials.csv')

def stats(df, method):
    ok = df[df['status'] == 'ok']['f1_macro_mean']
    pruned = (df['status'] == 'pruned').sum()
    return {
        'Método':      method,
        'Trials OK':   len(ok),
        'Podados':     pruned,
        'F1 mean':     ok.mean(),
        'F1 std':      ok.std(),
        'F1 max':      ok.max(),
        'F1 min':      ok.min(),
    }

summary = pd.DataFrame([stats(df_rand, 'Random'), stats(df_bayes, 'Bayesian (TPE)')])
summary = summary.set_index('Método')

print('=== Comparación de métodos ===')
display(summary.style.format({
    'F1 mean': '{:.4f}', 'F1 std': '{:.4f}',
    'F1 max':  '{:.4f}', 'F1 min': '{:.4f}',
}).highlight_max(subset=['F1 max', 'F1 mean'], color='#c3efb0'))

print('\n=== Distribución por modelo ===')
df_all = pd.concat([df_rand, df_bayes])
df_ok  = df_all[df_all['status'] == 'ok']
print(df_ok.groupby('model')['f1_macro_mean'].agg(['count','mean','std','max']).round(4))

## Config ganadora — entrenamiento final

In [ ]:
import re
from pathlib import Path

ROOT = Path('..') if Path('../data').exists() else Path('.')

resumen = (ROOT / 'data' / 'semana8_resumen.txt').read_text()

# Extraer sección CONFIG GANADORA
match = re.search(r'--- CONFIG GANADORA ---(.+)', resumen, re.DOTALL)
if match:
    print('=== CONFIG GANADORA ===')
    print(match.group(1).strip())

## Checklist Sprint 8

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..') if Path('../data').exists() else Path('.')

checks = [
    ('Experimentos comparables Random/Bayes',
     (ROOT/'data'/'semana8_random_trials.csv').exists() and
     (ROOT/'data'/'semana8_bayes_trials.csv').exists()),

    ('Early stopping en HGB (early_stopping=True)',
     True),

    ('Pruning en Bayes (MedianPruner)',
     True),

    ('Logs + artefactos guardados',
     (ROOT/'data'/'semana8_resumen.txt').exists()),

    ('Tabla top-k',
     True),

    ('Gráfico de evolución',
     (ROOT/'data'/'semana8_graficos.png').exists()),

    ('Resumen espacio/presupuesto/decisión',
     (ROOT/'data'/'semana8_resumen.txt').exists()),
]

for item, ok in checks:
    print(f"  {'[OK]' if ok else '[FALTA]'} {item}")

all_ok = all(ok for _, ok in checks)
print(f"\n{'[SPRINT 8 COMPLETADO]' if all_ok else '[INCOMPLETO]'}")